In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

In [ ]:
TRAIN_MODE   = 0
INF_MODE     = 1 # Inference / Predication Mode 
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 60 # only for scratch model
UPLOAD_MODEL = {'upload': TRAIN_MODE, 'file_name': 'model_t1', 'url': "jashmevada/dlgenai-nppe1/pytorch/fine_tune"}
LOAD_MODEL   = INF_MODE # Use Model from kagglehub
TRACKIO_NAME = "fine_2"
MODEL_TYPE   = 1 # 0 -> scratch cnn ; 1 -> fine tune 

In [ ]:
!pip install trackio -qq

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

In [ ]:
import trackio
if TRAIN_MODE: trackio.init(project="25-t3-nppe1", space_id="JashMevada/dlgenai-nppe", name=TRACKIO_NAME, group="fine_tune_cnn")

In [ ]:
df = pd.read_csv("/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset/train.csv")
test_df = pd.read_csv("/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset/test.csv")

# Scratch CNN Model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms
from PIL import Image
import os
import pytorch_lightning as pl
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms


class FaceDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.dataframe.iloc[idx]['full_path'])
        image = Image.open(img_path).convert('RGB')
        
        gender = self.dataframe.iloc[idx]['gender'] if 'gender' in self.dataframe.columns else -1
        age = self.dataframe.iloc[idx]['age'] if 'age' in self.dataframe.columns else -1
        
        if self.transform:
            image = self.transform(image)

        return image, gender, age

class FaceDataModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, img_dir, batch_size=32, num_workers=4):
        super().__init__()
        self.train_df = train_df
        self.test_df = test_df
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            full_dataset = FaceDataset(self.train_df, self.img_dir, transform=self.transform)
            train_size = int(0.8 * len(full_dataset))
            val_size = len(full_dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(full_dataset, [train_size, val_size])

        if stage == 'test' or stage is None:
            self.test_dataset = FaceDataset(self.test_df, self.img_dir, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, num_workers=self.num_workers, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=self.num_workers)



In [ ]:
img_dir = "/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset/"
face_data_module = FaceDataModule(train_df=df, test_df=test_df, img_dir=img_dir)

face_data_module.setup()

train_loader = face_data_module.train_dataloader()
test_loader = face_data_module.test_dataloader()
val_loader = face_data_module.val_dataloader()

images, genders, ages = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Gender batch shape:", genders.shape)
print("Age batch shape:", ages.shape)

In [ ]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torchmetrics
import torch.optim as optim
from torchmetrics import Metric

def calculate_gender_f1(true_genders, predicted_gender_probs):

    f1 = torchmetrics.F1Score(task="binary")
    return f1(predicted_gender_probs, true_genders.long())

def calculate_age_nrmse(true_ages, predicted_ages):

    rmse = torchmetrics.MeanSquaredError(squared=False).to(DEVICE)
    age_range = torch.max(true_ages) - torch.min(true_ages)
    if age_range == 0:
        return torch.tensor(float('inf')) # Avoid division by zero

    rmse_val = rmse(predicted_ages, true_ages)  # tensor scalar
    clipped = torch.clamp(rmse_val, max=torch.tensor(30.0, device=DEVICE))
    age_score = 1.0 - clipped / 30.0
    return age_score

class AgeScore(Metric):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.add_state("sum_squared_error", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("total", default=torch.tensor(0), dist_reduce_fx="sum")
    
    def update(self, preds: torch.Tensor, target: torch.Tensor):
        preds = preds.view(-1)
        target = target.view(-1)
        self.sum_squared_error += torch.sum((preds - target) ** 2)
        self.total += preds.numel()
    
    def compute(self):
        rmse = torch.sqrt(self.sum_squared_error / self.total)
        clipped_rmse = torch.minimum(rmse, torch.tensor(30.0, device=rmse.device))
        age_score = 1.0 - clipped_rmse / 30.0
        return age_score


class FaceModel(pl.LightningModule):
    def __init__(self):
        super().__init__()

        # Shared CNN Backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Flatten()
        )

        # Gender Prediction Head
        self.gender_head = nn.Sequential(
            nn.Linear(128 * 16 * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        # Age Prediction Head
        self.age_head = nn.Sequential(
            nn.Linear(128 * 16 * 16, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.gender_loss_fn = nn.BCEWithLogitsLoss()
        self.age_loss_fn = nn.MSELoss()

        # Metrics
        self.gender_f1 = torchmetrics.F1Score(task="binary")
        self.age_mae = torchmetrics.MeanAbsoluteError()
        self.age_rmse = torchmetrics.MeanSquaredError(squared=False)
        self.age_score = AgeScore()
        # Store outputs for epoch-level calculations
        self.validation_step_outputs = []
        self.test_step_outputs = []
        self.predict_step_outputs = []


    def forward(self, x):
        x = self.backbone(x)
        gender_pred = self.gender_head(x)
        age_pred = self.age_head(x)
        return gender_pred, age_pred

    def training_step(self, batch, batch_idx):
        images, genders, ages = batch

        gender_preds, age_preds = self(images)

        gender_loss = self.gender_loss_fn(gender_preds.squeeze(), genders.float())
        age_loss = self.age_loss_fn(age_preds.squeeze(), ages.float())

        total_loss = gender_loss + age_loss

        self.log('train_gender_loss', gender_loss)
        self.log('train_age_loss', age_loss)
        self.log('train_total_loss', total_loss)

        trackio.log({'train_gender_loss': gender_loss.item(), 'train_age_loss': age_loss.item(), 'train_total_loss': total_loss.item()})

        return total_loss

    def validation_step(self, batch, batch_idx):
        images, genders, ages = batch

        gender_preds, age_preds = self(images)

        # Calculate losses
        gender_loss = self.gender_loss_fn(gender_preds.squeeze(), genders.float())
        age_loss = self.age_loss_fn(age_preds.squeeze(), ages.float())
        total_loss = gender_loss + age_loss

        # Calculate gender F1 score
        gender_probs = torch.sigmoid(gender_preds.squeeze())
        gender_labels = genders.long()
        self.gender_f1.update(gender_probs, gender_labels)

        # Calculate age nRMSE
        self.age_mae.update(age_preds.squeeze(), ages.float())
        self.age_rmse.update(age_preds.squeeze(), ages.float())
        self.age_score.update(age_preds.squeeze(), ages.float())

        # Log losses and metrics
        self.log('val_gender_loss', gender_loss)
        self.log('val_age_loss', age_loss)
        self.log('val_total_loss', total_loss)
        self.log('val_gender_f1_step', self.gender_f1) 
        self.log('val_age_mae_step', self.age_mae)
        self.log('val_age_rmse_step', self.age_rmse) 
        self.log('val_age_score_step', self.age_score)

        trackio.log({'val_gender_loss': gender_loss.item(), 'val_age_rmse_step': self.age_rmse.compute().item(),'val_age_mae_step': self.age_mae.compute().item(),
                     'val_gender_f1_step': self.gender_f1.compute().item(), 'val_total_loss': total_loss.item(), 'val_age_loss': age_loss.item(), 'val_age_score_step': self.age_score.compute().item()})

        output = {'val_loss': total_loss, 'gender_preds': gender_preds, 'age_preds': age_preds, 'genders': genders, 'ages': ages}

        self.validation_step_outputs.append(output)
        return output


    def on_validation_epoch_end(self):
        outputs = self.validation_step_outputs
        
        epoch_gender_f1 = self.gender_f1.compute()
        self.log('val_gender_f1_epoch', epoch_gender_f1)
        trackio.log({'val_gender_f1_epoch': epoch_gender_f1.item()})
        self.gender_f1.reset()
        self.age_score.reset()
     
        all_ages = torch.cat([x['ages'] for x in outputs])
        all_age_preds = torch.cat([x['age_preds'].squeeze() for x in outputs])

        epoch_age_nrmse = calculate_age_nrmse(all_ages, all_age_preds)
        self.log('val_age_nrmse_epoch', epoch_age_nrmse)
        trackio.log({'val_age_nrmse_epoch': epoch_age_nrmse.item()})
        self.age_mae.reset() 
        self.age_rmse.reset() 
        self.age_score.reset()

        # Calculate harmonic mean
        if epoch_gender_f1 > 0 and not torch.isinf(epoch_age_nrmse):
            age_metric_for_harmonic_mean = 1.0 - epoch_age_nrmse

            age_metric_for_harmonic_mean = torch.max(torch.tensor(0.0), age_metric_for_harmonic_mean)

            if epoch_gender_f1 + age_metric_for_harmonic_mean > 0:
                 harmonic_mean = 2 * (epoch_gender_f1 * age_metric_for_harmonic_mean) / (epoch_gender_f1 + age_metric_for_harmonic_mean)
            else:
                 harmonic_mean = torch.tensor(0.0)
        else:
            harmonic_mean = torch.tensor(0.0)

        self.log('val_harmonic_mean', harmonic_mean)
        trackio.log({'val_harmonic_mean': harmonic_mean.item()})
        self.validation_step_outputs.clear()

    def predict_step(self, batch, batch_idx):
        images, _, _ = batch 

        gender_preds, age_preds = self(images)

        gender_probs = torch.sigmoid(gender_preds.squeeze())

        output = {'gender_probs': gender_probs, 'age_preds': age_preds.squeeze()}
        self.predict_step_outputs.append(output)
        return output


    def on_predict_epoch_end(self):
        outputs = self.predict_step_outputs

        all_gender_probs = torch.cat([x['gender_probs'] for x in outputs])
        all_age_preds = torch.cat([x['age_preds'] for x in outputs])

        self.predict_step_outputs.clear()

        return {'gender_probs': all_gender_probs, 'age_preds': all_age_preds}


    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [ ]:
img_dir = "/kaggle/input/sep-25-dl-gen-ai-nppe-1/face_dataset/"
face_data_module = FaceDataModule(train_df=df, test_df=test_df, img_dir=img_dir, batch_size=128, num_workers=4)

model = FaceModel()
trainer = pl.Trainer(max_epochs=EPOCHS,log_every_n_steps=50)

In [ ]:
def train():
    face_data_module.setup()
    trainer.fit(model, face_data_module)
    return trainer
    
if MODEL_TYPE == 0 and TRAIN_MODE: 
    print("Being Training...")
    trainer = train()
    trackio.finish()

In [ ]:
import kagglehub
def create_submission(model, data_module: pl.LightningDataModule, test_df, output_path="submission.csv"):

    if LOAD_MODEL:
        path = kagglehub.model_download(UPLOAD_MODEL['url'])
        state_dict = torch.load(f"{path}/{UPLOAD_MODEL['file_name']}.pth")
        model.load_state_dict(state_dict)
    
    model.eval()
    data_module.setup()
    trainer = pl.Trainer()
    predictions = trainer.predict(model, dataloaders=data_module.test_dataloader())

    gender_probs = torch.cat([p['gender_probs'] for p in predictions])
    age_preds = torch.cat([p['age_preds'] for p in predictions])

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'gender': (gender_probs > 0.5).long().cpu().numpy(), 
        'age': age_preds.cpu().numpy().round().astype(int) 
    })

    submission_df.to_csv(output_path, index=False)
    print(f"Submission file saved to {output_path}")

if MODEL_TYPE == 0 and INF_MODE: create_submission(model, face_data_module, test_df)

# Fine Tune Model

In [ ]:
class FineTuneFaceDataModule(pl.LightningDataModule):
    def __init__(self, train_df, test_df, img_dir, batch_size=32, num_workers=4):
        super().__init__()
        self.train_df = train_df
        self.test_df = test_df
        self.img_dir = img_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

        # Modified transformations for ResNet50
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)), # Resize to 224x224
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet normalization
        ])

    def setup(self, stage=None):
        if stage == 'fit' or stage is None:
            full_dataset = FaceDataset(self.train_df, self.img_dir, transform=self.transform)
            train_size = int(0.8 * len(full_dataset))
            val_size = len(full_dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(full_dataset, [train_size, val_size])

        if stage == 'test' or stage is None:
            self.test_dataset = FaceDataset(self.test_df, self.img_dir, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, num_workers=self.num_workers, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=self.num_workers)


In [ ]:
import torchvision.models as models

class FaceModelResNet(pl.LightningModule):
    def __init__(self):
        super().__init__()

        resnet50 = models.resnet50(pretrained=True)

        self.backbone = nn.Sequential(*list(resnet50.children())[:-1])

        fc_input_features = 2048

        # Gender Prediction Head
        self.gender_head = nn.Sequential(
            nn.Linear(fc_input_features, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        # Age Prediction Head
        self.age_head = nn.Sequential(
            nn.Linear(fc_input_features, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.gender_loss_fn = nn.BCEWithLogitsLoss()
        self.age_loss_fn = nn.MSELoss()

        # Metrics
        self.gender_f1 = torchmetrics.F1Score(task="binary")
        self.age_rmse = torchmetrics.MeanSquaredError(squared=False).to(DEVICE)
        self.age_score = AgeScore()

        # Store outputs for epoch-level calculations
        self.validation_step_outputs = []
        self.test_step_outputs = []
        self.predict_step_outputs = []

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1) # Flatten the output from the backbone
        gender_pred = self.gender_head(x)
        age_pred = self.age_head(x)
        return gender_pred, age_pred

    def training_step(self, batch, batch_idx):
        images, genders, ages = batch # Unpack all four values

        gender_preds, age_preds = self(images)

        gender_loss = self.gender_loss_fn(gender_preds.squeeze(), genders.float())
        age_loss = self.age_loss_fn(age_preds.squeeze(), ages.float())

        total_loss = gender_loss + age_loss

        self.log('train_gender_loss', gender_loss)
        self.log('train_age_loss', age_loss)
        self.log('train_total_loss', total_loss)

        trackio.log({'train_gender_loss': gender_loss.item(), 'train_age_loss': age_loss.item(), 'train_total_loss': total_loss.item()})

        return total_loss

    def validation_step(self, batch, batch_idx):
        images, genders, ages = batch

        gender_preds, age_preds = self(images)

        gender_loss = self.gender_loss_fn(gender_preds.squeeze(), genders.float())
        age_loss = self.age_loss_fn(age_preds.squeeze(), ages.float())
        total_loss = gender_loss + age_loss

        gender_probs = torch.sigmoid(gender_preds.squeeze())
        gender_labels = genders.long()
        self.gender_f1.update(gender_probs, gender_labels)

        self.age_rmse.update(age_preds.squeeze(), ages.float())
        self.age_score.update(age_preds.squeeze(), ages.float())

        self.log('val_gender_loss', gender_loss)
        self.log('val_age_loss', age_loss)
        self.log('val_total_loss', total_loss)
        self.log('val_gender_f1_step', self.gender_f1)
        self.log('val_age_rmse_step', self.age_rmse)

        trackio.log({'val_gender_loss': gender_loss.item(), 'val_age_rmse_step': self.age_rmse.compute().item(),
                     'val_gender_f1_step': self.gender_f1.compute().item(), 'val_total_loss': total_loss.item(), 'val_age_loss': age_loss.item(), 'val_age_score_step': self.age_score.compute().item()})

        
        output = {'val_loss': total_loss, 'gender_preds': gender_preds, 'age_preds': age_preds, 'genders': genders, 'ages': ages}
        self.validation_step_outputs.append(output)
        return output


    def on_validation_epoch_end(self):
        outputs = self.validation_step_outputs
        epoch_gender_f1 = self.gender_f1.compute()
        self.log('val_gender_f1_epoch', epoch_gender_f1)
        trackio.log({'val_gender_f1_epoch': epoch_gender_f1.item()})
        
        self.gender_f1.reset() # Reset metric state

        all_ages = torch.cat([x['ages'] for x in outputs])
        all_age_preds = torch.cat([x['age_preds'].squeeze() for x in outputs])

        epoch_age_nrmse = calculate_age_nrmse(all_ages, all_age_preds)
        
        self.log('val_age_nrmse_epoch', epoch_age_nrmse)
        trackio.log({'val_age_nrmse_epoch': epoch_age_nrmse.item()})

        self.age_rmse.reset() # Reset metric state
        self.age_score.reset()

        if epoch_gender_f1 > 0 and not torch.isinf(epoch_age_nrmse):
            age_metric_for_harmonic_mean = 1.0 - epoch_age_nrmse
            age_metric_for_harmonic_mean = torch.max(torch.tensor(0.0), age_metric_for_harmonic_mean)

            if epoch_gender_f1 + age_metric_for_harmonic_mean > 0:
                 harmonic_mean = 2 * (epoch_gender_f1 * age_metric_for_harmonic_mean) / (epoch_gender_f1 + age_metric_for_harmonic_mean)
            else:
                 harmonic_mean = torch.tensor(0.0)
        else:
            harmonic_mean = torch.tensor(0.0) # Or some other appropriate value

        self.log('val_harmonic_mean', harmonic_mean)
        trackio.log({'val_harmonic_mean': harmonic_mean.item()})
        
        self.validation_step_outputs.clear()

    def predict_step(self, batch, batch_idx):
        images, _, _ = batch # We don't have gender or age labels for prediction

        gender_preds, age_preds = self(images)

        gender_probs = torch.sigmoid(gender_preds.squeeze())

        output = {'gender_probs': gender_probs, 'age_preds': age_preds.squeeze()}
        self.predict_step_outputs.append(output)
        return output


    def on_predict_epoch_end(self):
        outputs = self.predict_step_outputs
        all_gender_probs = torch.cat([x['gender_probs'] for x in outputs])
        all_age_preds = torch.cat([x['age_preds'] for x in outputs])
        self.predict_step_outputs.clear()

        return {'gender_probs': all_gender_probs, 'age_preds': all_age_preds}

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

In [ ]:
face_data_module_resnet = FineTuneFaceDataModule(train_df=df, test_df=test_df, img_dir=img_dir, batch_size=64, num_workers=4)

model_resnet = FaceModelResNet()

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning import Trainer

trainer_resnet = Trainer(
    max_epochs=15,
    log_every_n_steps=50,
    callbacks=[
        ModelCheckpoint(
            monitor='val_harmonic_mean',
            mode='max',
            save_top_k=1,
            filename='best-resnet-model-{epoch:02d}-{val_harmonic_mean:.4f}'
        ),
        EarlyStopping(
            monitor='val_harmonic_mean',
            mode='max',
            patience=3
        )
    ]
)

if MODEL_TYPE == 1 and TRAIN_MODE: 
    print("Begin Traning...")
    trainer_resnet.fit(model_resnet, face_data_module_resnet)
    trackio.finish()

In [ ]:
import kagglehub
def save_model(model):
    torch.save(model.state_dict(), f"/kaggle/working/{UPLOAD_MODEL['file_name']}.pth")
    kagglehub.model_upload(UPLOAD_MODEL['url'], f"/kaggle/working/{UPLOAD_MODEL['file_name']}.pth") 

if MODEL_TYPE == 1 and UPLOAD_MODEL['upload']:
    save_model(model_resnet)

if MODEL_TYPE == 0 and UPLOAD_MODEL['upload']:
    save_model(model)

In [ ]:
if INF_MODE and MODEL_TYPE == 1: create_submission(model_resnet, face_data_module_resnet, test_df)